# Evaluate & Record: Test Your Trained Agent

Load a trained IMPALA checkpoint, run evaluation episodes, and record videos.

**Requirements**: `pip install torch matplotlib imageio[ffmpeg]`  
**Time**: ~5 minutes  
**Prerequisite**: A trained checkpoint (from `02_train_and_plot.ipynb` or a full training run)

In [ ]:
%matplotlib inline

In [ ]:
import sys, os

import pathlib
PROJECT_ROOT = str(pathlib.Path.cwd().parent) if pathlib.Path('../train_impala.py').exists() else str(pathlib.Path.cwd())
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'notebooks'))
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
else:
    os.environ.setdefault("MUJOCO_GL", "glfw")

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from utils import record_episode, save_video, display_frame


def make_env(env_id, **kwargs):
    """Create env with EGL fallback — if MuJoCo fails due to EGL conflict, use Genesis."""
    try:
        return gym.make(env_id, **kwargs)
    except Exception as e:
        err_str = str(e)
        if "EGL" in err_str or "GLFWError" in err_str or "gladLoadGL" in err_str:
            genesis_id = env_id.replace("-v0", "-Genesis-v0")
            print(f"MuJoCo rendering unavailable, using Genesis: {genesis_id}")
            return gym.make(genesis_id, use_batch_renderer=False, **kwargs)
        raise

## 1. Load the Model

Import the IMPALA network from `train_impala.py` (no code duplication).

In [ ]:
from train_impala import MemoryMazeNet

print(f"MemoryMazeNet imported from {PROJECT_ROOT}/train_impala.py")

## 2. Load Checkpoint

Set the experiment ID below to match your training run.

In [ ]:
# Set this to your experiment ID from training
XPID = "impala_9x9_demo"  # from 02_train_and_plot.ipynb
SAVEDIR = os.path.expanduser("~/logs/torchbeast")

checkpointpath = os.path.join(SAVEDIR, XPID, "model.tar")

if not os.path.exists(checkpointpath):
    print(f"Checkpoint not found at: {checkpointpath}")
    print(f"\nAvailable experiments in {SAVEDIR}:")
    if os.path.exists(SAVEDIR):
        for d in sorted(os.listdir(SAVEDIR)):
            model_path = os.path.join(SAVEDIR, d, "model.tar")
            if os.path.exists(model_path):
                size_mb = os.path.getsize(model_path) / 1e6
                print(f"  {d}  ({size_mb:.1f} MB)")
    else:
        print(f"  (directory does not exist)")
    print(f"\nRun 02_train_and_plot.ipynb first, or set XPID to an existing experiment.")
else:
    checkpoint = torch.load(checkpointpath, map_location="cpu")
    print(f"Loaded checkpoint: {checkpointpath}")
    print(f"  Training step: {checkpoint.get('step', 'unknown'):,}")
    print(f"  Keys: {list(checkpoint.keys())}")

In [ ]:
model = MemoryMazeNet(observation_shape=(3, 64, 64), num_actions=6)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")

## 3. Run Evaluation Episodes

Run the agent for multiple episodes and collect returns.

In [ ]:
env = make_env("memory_maze:MemoryMaze-9x9-v0", disable_env_checker=True)

NUM_EPISODES = 10
returns = []
lengths = []

for ep in range(NUM_EPISODES):
    obs = env.reset()
    core_state = model.initial_state(batch_size=1)
    last_action = torch.zeros(1, dtype=torch.long)
    reward_t = torch.zeros(1)
    episode_return = 0.0
    episode_steps = 0

    done = False
    while not done:
        obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)  # [1,1,C,H,W]
        inputs = {
            "frame": obs_t,
            "last_action": last_action.view(1, 1),
            "reward": reward_t.view(1, 1),
        }
        with torch.no_grad():
            outputs, core_state = model(inputs, core_state)

        action = outputs["action"].item()
        obs, reward, done, info = env.step(action)
        last_action = torch.tensor([action])
        reward_t = torch.tensor([reward])
        episode_return += reward
        episode_steps += 1

    returns.append(episode_return)
    lengths.append(episode_steps)
    print(f"  Episode {ep+1:2d}: return={episode_return:5.1f}, steps={episode_steps}")

print(f"\n--- Summary ({NUM_EPISODES} episodes) ---")
print(f"  Mean return:  {np.mean(returns):.1f} +/- {np.std(returns):.1f}")
print(f"  Min/Max:      {np.min(returns):.1f} / {np.max(returns):.1f}")
print(f"  Mean length:  {np.mean(lengths):.0f}")

## 4. Record an Episode as Video

Record the agent playing an episode and save as MP4.

In [ ]:
def make_policy_fn(model):
    """Create a policy function for record_episode (accepts obs + reward)."""
    core_state = model.initial_state(batch_size=1)
    last_action = torch.zeros(1, dtype=torch.long)
    reward_t = torch.zeros(1)

    def policy(obs, reward=0.0):
        nonlocal core_state, last_action, reward_t
        reward_t = torch.tensor([reward])
        obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)
        inputs = {
            "frame": obs_t,
            "last_action": last_action.view(1, 1),
            "reward": reward_t.view(1, 1),
        }
        with torch.no_grad():
            outputs, core_state_new = model(inputs, core_state)
        core_state = core_state_new
        action = outputs["action"].item()
        last_action = torch.tensor([action])
        return action

    return policy


frames, total_return, num_steps = record_episode(
    env, policy_fn=make_policy_fn(model), max_steps=1000, fps=10
)
print(f"Recorded {len(frames)} frames, return={total_return:.1f}, steps={num_steps}")

In [ ]:
save_video(frames, "agent_episode.mp4", fps=10)
print(f"Saved: agent_episode.mp4 ({len(frames)} frames)")

# Display sample frames from the episode
indices = np.linspace(0, len(frames) - 1, 6, dtype=int)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, idx in zip(axes, indices):
    ax.imshow(frames[idx])
    ax.set_title(f"Step {idx}")
    ax.axis("off")
plt.suptitle(f"Agent Episode (return={total_return:.1f})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
try:
    from IPython.display import Video, display
    display(Video("agent_episode.mp4", embed=True, html_attributes="loop autoplay muted"))
except ImportError:
    print("Run in Jupyter to see inline video, or open agent_episode.mp4 directly")

## 5. Compare Random vs Trained Agent

In [ ]:
# Random agent
frames_random, return_random, steps_random = record_episode(env, policy_fn=None, max_steps=200)

# Trained agent
frames_trained, return_trained, steps_trained = record_episode(
    env, policy_fn=make_policy_fn(model), max_steps=200)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col, idx in enumerate(np.linspace(0, min(len(frames_random), len(frames_trained)) - 1, 5, dtype=int)):
    axes[0, col].imshow(frames_random[min(idx, len(frames_random)-1)])
    axes[0, col].set_title(f"Step {idx}")
    axes[0, col].axis("off")
    axes[1, col].imshow(frames_trained[min(idx, len(frames_trained)-1)])
    axes[1, col].set_title(f"Step {idx}")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Random\nAgent", fontsize=12, rotation=0, labelpad=60)
axes[1, 0].set_ylabel("Trained\nAgent", fontsize=12, rotation=0, labelpad=60)
plt.suptitle(f"Random (return={return_random:.1f}) vs Trained (return={return_trained:.1f})", fontsize=14)
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
env.close()
print("Done! Next: 04_compare_algorithms.ipynb")